In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from typing import Dict, Tuple, Callable, List, Optional

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEQUENCE_LENGTH = 252
WINDOW = 30
REGIME_NAMES = ["low_vol", "mid_vol", "high_vol"]
N_REGIMES = 3

In [2]:
pip install arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 59.4 MB/s eta 0:00:00


In [3]:
def fetch_vix_data(
    start: str = "2005-01-01", end: str = "2025-12-31"
) -> Tuple[pd.Series, pd.Series]:
    vix = yf.download("^VIX", start=start, end=end, progress=False)["Close"].squeeze()
    tnx = yf.download("^TNX", start=start, end=end, progress=False)["Close"].squeeze()
    vix.name, tnx.name = "VIX", "TNX"
    return vix, tnx


def align_and_fill(
    vix: pd.Series, tnx: pd.Series, ffill_limit: int = 3, max_missing_frac: float = 0.02
) -> Tuple[pd.Series, pd.Series]:
    common_idx = vix.index.union(tnx.index).sort_values()
    vix = vix.reindex(common_idx).ffill(limit=ffill_limit)
    tnx = tnx.reindex(common_idx).ffill(limit=ffill_limit)
    mask = vix.notna() & tnx.notna()
    missing_frac = 1.0 - mask.mean()
    if missing_frac > max_missing_frac:
        raise ValueError(
            f"DataQualityError: {missing_frac:.2%} missing after alignment (limit {max_missing_frac:.0%})"
        )
    vix, tnx = vix[mask], tnx[mask]
    return vix, tnx


def build_vix_features(
    vix: pd.Series, tnx: pd.Series
) -> Tuple[pd.DataFrame, np.ndarray, StandardScaler, pd.DatetimeIndex]:
    vix_ema = vix.ewm(span=5, adjust=False).mean()
    vix_21d_chg = vix_ema.pct_change(periods=21)
    tnx_21d_chg = tnx.diff(periods=21)

    features = pd.DataFrame(
        {"vix_ema": vix_ema, "vix_21d_chg": vix_21d_chg, "tnx_21d_chg": tnx_21d_chg}
    )
    features = features.iloc[21:]
    features = features.dropna()

    scaler = StandardScaler()
    X = scaler.fit_transform(features.values)
    return features, X, scaler, features.index


def fetch_vix_pipeline(
    start: str = "2005-01-01", end: str = "2025-12-31"
) -> Tuple[pd.Series, pd.DataFrame, np.ndarray, StandardScaler, pd.DatetimeIndex]:
    """End-to-end VIX pipeline: fetch, align, build features, scale."""
    vix_raw, tnx_raw = fetch_vix_data(start, end)
    vix, tnx = align_and_fill(vix_raw, tnx_raw)
    features_df, X, scaler, dates = build_vix_features(vix, tnx)
    return vix, features_df, X, scaler, dates

In [4]:
TICKERS = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "JPM", "BAC",
    "XOM", "CVX", "JNJ", "PG", "UNH", "HD", "KO", "PEP", "WMT",
]


def fetch_equities_returns(
    tickers: List[str] = TICKERS,
    start: str = "2005-01-01",
    end: str = "2025-12-31",
) -> pd.DataFrame:
    prices = yf.download(tickers, start=start, end=end, progress=False)["Close"]
    if isinstance(prices, pd.Series):
        prices = prices.to_frame()
    prices = prices.dropna(how="any")
    log_returns = np.log(prices / prices.shift(1)).iloc[1:]
    log_returns = log_returns.dropna(how="any")
    return log_returns

In [5]:
def fit_regime_gmm(
    X: np.ndarray,
    features_df: pd.DataFrame,
    dates: pd.DatetimeIndex,
) -> Tuple[GaussianMixture, pd.Series]:
    gmm = GaussianMixture(
        n_components=N_REGIMES,
        covariance_type="full",
        init_params="k-means++",
        max_iter=300,
        tol=1e-4,
        n_init=3,
        reg_covar=1e-6,
        random_state=42,
    )
    gmm.fit(X)

    raw_labels = gmm.predict(X)

    vix_ema_values = features_df["vix_ema"].values
    component_mean_vix = np.array(
        [vix_ema_values[raw_labels == c].mean() for c in range(N_REGIMES)]
    )
    sorted_order = np.argsort(component_mean_vix)
    index_to_regime = {sorted_order[i]: REGIME_NAMES[i] for i in range(N_REGIMES)}

    regime_labels = pd.Series(
        [index_to_regime[l] for l in raw_labels], index=dates, name="regime"
    )
    return gmm, regime_labels


def smooth_regime_labels(labels: pd.Series, min_run: int = 5) -> pd.Series:
    smoothed = labels.copy()
    vals = smoothed.values
    n = len(vals)
    i = 0
    while i < n:
        j = i
        while j < n and vals[j] == vals[i]:
            j += 1
        run_len = j - i
        if run_len < min_run:
            left = vals[i - 1] if i > 0 else vals[j] if j < n else vals[i]
            right = vals[j] if j < n else vals[i - 1] if i > 0 else vals[i]
            replacement = left if left == right else left  # majority of surrounding
            vals[i:j] = replacement
        i = j
    smoothed[:] = vals
    return smoothed


In [6]:
def estimate_transition_matrix(labels: pd.Series) -> np.ndarray:
    regime_to_idx = {name: i for i, name in enumerate(REGIME_NAMES)}
    indices = np.array([regime_to_idx[l] for l in labels if l in regime_to_idx])
    T = np.zeros((N_REGIMES, N_REGIMES))
    for t in range(len(indices) - 1):
        T[indices[t], indices[t + 1]] += 1
    row_sums = T.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    T = T / row_sums
    return T


def make_markov_step(transition_matrix: np.ndarray) -> Callable[[int], int]:
    def step(current_regime: int) -> int:
        return np.random.choice(N_REGIMES, p=transition_matrix[current_regime])
    return step


def generate_regime_sequence(
    transition_matrix: np.ndarray, initial_regime: int, length: int = SEQUENCE_LENGTH
) -> np.ndarray:
    step = make_markov_step(transition_matrix)
    regimes = np.empty(length, dtype=int)
    regimes[0] = initial_regime
    for t in range(1, length):
        regimes[t] = step(regimes[t - 1])
    return regimes

In [7]:
def build_training_dataset(
    returns_df: pd.DataFrame,
    regime_labels: pd.Series,
    window: int = WINDOW,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    regime_to_idx = {name: i for i, name in enumerate(REGIME_NAMES)}
    common_dates = returns_df.index.intersection(regime_labels.index)
    returns_aligned = returns_df.loc[common_dates]
    regimes_aligned = regime_labels.loc[common_dates]

    X_windows: List[np.ndarray] = []
    X_regimes: List[int] = []
    y_targets: List[float] = []

    for col in returns_aligned.columns:
        r = returns_aligned[col].values
        reg = regimes_aligned.values
        for t in range(window, len(r) - 1):
            label = reg[t]
            if label not in regime_to_idx:
                continue
            X_windows.append(r[t - window : t])
            X_regimes.append(regime_to_idx[label])
            y_targets.append(r[t + 1])

    return (
        np.array(X_windows, dtype=np.float32),
        np.array(X_regimes, dtype=np.int64),
        np.array(y_targets, dtype=np.float32),
    )

In [8]:
def train_gbm(
    returns_df: pd.DataFrame, regime_labels: pd.Series
) -> Dict[str, Tuple[float, float]]:
    dt = 1.0 / 252.0
    regime_to_idx = {name: i for i, name in enumerate(REGIME_NAMES)}
    common = returns_df.index.intersection(regime_labels.index)
    returns_aligned = returns_df.loc[common]
    regimes_aligned = regime_labels.loc[common]

    params: Dict[str, Tuple[float, float]] = {}
    for regime_name in REGIME_NAMES:
        mask = regimes_aligned == regime_name
        pooled = returns_aligned.loc[mask].values.ravel()
        pooled = pooled[np.isfinite(pooled)]
        mu = pooled.mean() / dt
        sigma = pooled.std() / np.sqrt(dt)
        params[regime_name] = (mu, sigma)
    return params


def gbm_generate_next(
    window: np.ndarray, regime: int, gbm_params: Dict[str, Tuple[float, float]]
) -> float:
    dt = 1.0 / 252.0
    mu, sigma = gbm_params[REGIME_NAMES[regime]]
    return mu * dt + sigma * np.sqrt(dt) * np.random.randn()

In [9]:
def train_garch(
    returns_df: pd.DataFrame, regime_labels: pd.Series
) -> Dict[str, Dict[str, float]]:

    from arch import arch_model

    common = returns_df.index.intersection(regime_labels.index)
    returns_aligned = returns_df.loc[common]
    regimes_aligned = regime_labels.loc[common]

    params: Dict[str, Dict[str, float]] = {}
    for regime_name in REGIME_NAMES:
        mask = regimes_aligned == regime_name
        pooled = returns_aligned.loc[mask].values.ravel()
        pooled = pooled[np.isfinite(pooled)]

        am = arch_model(
            pooled * 100, vol="Garch", p=1, q=1, dist="StudentsT", mean="Zero"
        )
        res = am.fit(disp="off", show_warning=False)
        alpha_val = res.params.get("alpha[1]", 0.05)
        beta_val = res.params.get("beta[1]", 0.90)
        if alpha_val + beta_val >= 1.0:
            beta_val = 0.999 - alpha_val
        params[regime_name] = {
            "omega": res.params["omega"] / 1e4,
            "alpha": alpha_val,
            "beta": beta_val,
            "nu": res.params.get("nu", 30.0),
        }
    return params


def garch_generate_next(
    window: np.ndarray, regime: int, garch_params: Dict[str, Dict[str, float]]
) -> float:
    """Step GARCH(1,1) forward one day using the 30-day window to warm up variance."""
    p = garch_params[REGIME_NAMES[regime]]
    omega, alpha, beta, nu = p["omega"], p["alpha"], p["beta"], p["nu"]

    sigma2 = np.var(window)
    for r in window:
        sigma2 = omega + alpha * r**2 + beta * sigma2

    eps = np.random.standard_t(df=max(nu, 2.01))
    return np.sqrt(sigma2) * eps


In [10]:
def train_heston(
    returns_df: pd.DataFrame, regime_labels: pd.Series
) -> Dict[str, Dict[str, float]]:
    """Calibrate Heston parameters per regime via method of moments."""
    common = returns_df.index.intersection(regime_labels.index)
    returns_aligned = returns_df.loc[common]
    regimes_aligned = regime_labels.loc[common]

    params: Dict[str, Dict[str, float]] = {}
    for regime_name in REGIME_NAMES:
        mask = regimes_aligned == regime_name
        pooled = returns_aligned.loc[mask].values.ravel()
        pooled = pooled[np.isfinite(pooled)]

        mu = float(np.mean(pooled)) * 252
        daily_var = pooled**2
        theta = float(np.mean(daily_var)) * 252
        theta = np.clip(theta, 1e-6, 0.5)

        # kappa from autocorrelation of daily squared returns
        if len(daily_var) > 2:
            ac1 = np.corrcoef(daily_var[:-1], daily_var[1:])[0, 1]
            ac1 = np.clip(ac1, 0.001, 0.999)
            kappa = float(-252 * np.log(ac1))
        else:
            kappa = 5.0
        kappa = np.clip(kappa, 0.1, 20.0)

        xi = float(np.std(daily_var)) * np.sqrt(252) * 2.0 / max(np.sqrt(theta), 1e-4)
        xi = np.clip(xi, 0.05, 2.0)

        if len(pooled) > 2:
            dvar = np.diff(daily_var)
            rho = float(np.corrcoef(pooled[:-1], dvar)[0, 1])
            rho = np.clip(rho, -0.99, 0.99)
        else:
            rho = -0.7

        # Enforce Feller condition
        if 2 * kappa * theta <= xi**2:
            xi = np.sqrt(2 * kappa * theta * 0.95)

        params[regime_name] = {
            "mu": mu,
            "kappa": kappa,
            "theta": theta,
            "xi": xi,
            "rho": rho,
            "v0": theta,
        }
    return params


def heston_generate_next(
    window: np.ndarray, regime: int, heston_params: Dict[str, Dict[str, float]]
) -> float:
    p = heston_params[REGIME_NAMES[regime]]
    mu, kappa, theta, xi, rho = p["mu"], p["kappa"], p["theta"], p["xi"], p["rho"]
    dt = 1.0 / 252.0

    v_t = float(np.var(window[-5:])) * 252 if len(window) >= 5 else p["v0"]
    v_t = max(v_t, 1e-8)

    z1 = np.random.randn()
    z2 = rho * z1 + np.sqrt(1 - rho**2) * np.random.randn()

    v_pos = max(v_t, 0.0)
    dv = kappa * (theta - v_pos) * dt + xi * np.sqrt(v_pos * dt) * z2
    v_next = max(v_t + dv, 1e-8)
    r = mu * dt + np.sqrt(v_pos * dt) * z1
    return float(r)

In [11]:
class MixtureOfLogisticsHead(nn.Module):
    def __init__(self, d_in: int, n_components: int = 5):
        super().__init__()
        self.n_components = n_components
        self.proj = nn.Linear(d_in, n_components * 3)  # means, log_scales, logits

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        out = self.proj(x)
        means = out[:, : self.n_components]
        log_scales = out[:, self.n_components : 2 * self.n_components]
        logits = out[:, 2 * self.n_components :]
        return means, log_scales, logits

    @staticmethod
    def nll(
        y: torch.Tensor,
        means: torch.Tensor,
        log_scales: torch.Tensor,
        logits: torch.Tensor,
    ) -> torch.Tensor:
        y = y.unsqueeze(-1)
        centered = y - means
        s = torch.exp(log_scales) + 1e-8
        inv_s = 1.0 / s
        log_weights = logits - torch.logsumexp(logits, dim=-1, keepdim=True)
        log_pdf = -log_scales - centered * inv_s - 2.0 * F.softplus(-centered * inv_s)
        return -torch.logsumexp(log_pdf + log_weights, dim=-1).mean()

    def sample(self, means: torch.Tensor, log_scales: torch.Tensor, logits: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(logits, dim=-1)
        comp = torch.multinomial(weights, 1).squeeze(-1)
        mu = means[torch.arange(len(comp)), comp]
        s = torch.exp(log_scales[torch.arange(len(comp)), comp]) + 1e-8
        u = torch.rand_like(mu).clamp(1e-6, 1 - 1e-6)
        return mu + s * torch.log(u / (1 - u))


class TFTModel(nn.Module):
    def __init__(
        self,
        window: int = WINDOW,
        d_model: int = 64,
        d_regime: int = 16,
        n_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.1,
        n_mix: int = 5,
    ):
        super().__init__()
        self.window = window
        self.d_total = d_model + d_regime

        self.return_proj = nn.Linear(1, d_model)
        self.regime_proj = nn.Linear(N_REGIMES, d_regime)
        self.pos_embed = nn.Parameter(torch.randn(1, window, self.d_total) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_total,
            nhead=n_heads,
            dim_feedforward=self.d_total * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = MixtureOfLogisticsHead(self.d_total, n_mix)

    def forward(
        self, windows: torch.Tensor, regimes: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, T = windows.shape
        ret_emb = self.return_proj(windows.unsqueeze(-1))  # (B, T, d_model)
        reg_oh = F.one_hot(regimes, N_REGIMES).float().unsqueeze(1).expand(-1, T, -1)
        reg_emb = self.regime_proj(reg_oh)  # (B, T, d_regime)
        x = torch.cat([ret_emb, reg_emb], dim=-1) + self.pos_embed[:, :T, :]
        x = self.encoder(x)
        out = x[:, -1, :]  # last timestep
        return self.head(out)


def train_tft(
    X_windows: np.ndarray,
    X_regimes: np.ndarray,
    y_targets: np.ndarray,
    epochs: int = 50,
    batch_size: int = 256,
    patience: int = 10,
    lr: float = 1e-3,
) -> Tuple[TFTModel, float]:
    n = len(y_targets)
    split = int(0.85 * n)
    ds_train = TensorDataset(
        torch.tensor(X_windows[:split]),
        torch.tensor(X_regimes[:split]),
        torch.tensor(y_targets[:split]),
    )
    ds_val = TensorDataset(
        torch.tensor(X_windows[split:]),
        torch.tensor(X_regimes[split:]),
        torch.tensor(y_targets[split:]),
    )
    loader_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True)
    loader_val = DataLoader(ds_val, batch_size=batch_size)

    model = TFTModel().to(DEVICE)
    optim_ = torch.optim.Adam(model.parameters(), lr=lr)
    best_val, wait, best_state = float("inf"), 0, None

    for epoch in range(epochs):
        model.train()
        for xw, xr, yt in loader_train:
            xw, xr, yt = xw.to(DEVICE), xr.to(DEVICE), yt.to(DEVICE)
            means, log_scales, logits = model(xw, xr)
            loss = MixtureOfLogisticsHead.nll(yt, means, log_scales, logits)
            optim_.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim_.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xw, xr, yt in loader_val:
                xw, xr, yt = xw.to(DEVICE), xr.to(DEVICE), yt.to(DEVICE)
                m, ls, lo = model(xw, xr)
                val_loss += MixtureOfLogisticsHead.nll(yt, m, ls, lo).item()
        val_loss /= max(len(loader_val), 1)

        if val_loss < best_val:
            best_val, wait = val_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

    residuals = []
    with torch.no_grad():
        for xw, xr, yt in loader_val:
            xw, xr, yt = xw.to(DEVICE), xr.to(DEVICE), yt.to(DEVICE)
            m, ls, lo = model(xw, xr)
            # Use mixture mean as point prediction
            weights = torch.softmax(lo, dim=-1)
            pred = (weights * m).sum(dim=-1)
            residuals.append((yt - pred).cpu().numpy())
    residual_std = float(np.std(np.concatenate(residuals))) if residuals else 1e-4
    return model, residual_std


def tft_generate_next(
    window: np.ndarray, regime: int, tft_model: TFTModel, residual_std: float
) -> float:
    tft_model.eval()
    with torch.no_grad():
        xw = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        xr = torch.tensor([regime], dtype=torch.long).to(DEVICE)
        means, log_scales, logits = tft_model(xw, xr)
        sample = tft_model.head.sample(means, log_scales, logits)
    noise = np.random.randn() * residual_std * 0.1
    return float(sample.cpu().item()) + noise

In [12]:
class CVAEModel(nn.Module):
    def __init__(
        self,
        window: int = WINDOW,
        latent_dim: int = 16,
        hidden_dim: int = 128,
    ):
        super().__init__()
        self.latent_dim = latent_dim
        enc_in = window + N_REGIMES + 1  # window + regime_onehot + target
        self.encoder = nn.Sequential(
            nn.Linear(enc_in, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.enc_mu = nn.Linear(hidden_dim, latent_dim)
        self.enc_logvar = nn.Linear(hidden_dim, latent_dim)

        dec_in = latent_dim + window + N_REGIMES
        self.decoder = nn.Sequential(
            nn.Linear(dec_in, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def encode(
        self, window: torch.Tensor, regime_oh: torch.Tensor, target: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        x = torch.cat([window, regime_oh, target.unsqueeze(-1)], dim=-1)
        h = self.encoder(x)
        return self.enc_mu(h), self.enc_logvar(h)

    def reparameterise(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(
        self, z: torch.Tensor, window: torch.Tensor, regime_oh: torch.Tensor
    ) -> torch.Tensor:
        x = torch.cat([z, window, regime_oh], dim=-1)
        return self.decoder(x).squeeze(-1)

    def forward(
        self, window: torch.Tensor, regime: torch.Tensor, target: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        regime_oh = F.one_hot(regime, N_REGIMES).float()
        mu, logvar = self.encode(window, regime_oh, target)
        z = self.reparameterise(mu, logvar)
        recon = self.decode(z, window, regime_oh)
        return recon, mu, logvar


def cvae_loss(
    recon: torch.Tensor,
    target: torch.Tensor,
    mu: torch.Tensor,
    logvar: torch.Tensor,
    beta: float = 4.0,
) -> Tuple[torch.Tensor, float]:
    recon_loss = F.mse_loss(recon, target, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl, kl.item()


def train_cvae(
    X_windows: np.ndarray,
    X_regimes: np.ndarray,
    y_targets: np.ndarray,
    epochs: int = 60,
    batch_size: int = 256,
    beta: float = 4.0,
    lr: float = 1e-3,
    patience: int = 10,
) -> Tuple[CVAEModel, float]:
    """Train CVAE; returns model and residual scale."""
    n = len(y_targets)
    split = int(0.85 * n)
    ds_train = TensorDataset(
        torch.tensor(X_windows[:split]),
        torch.tensor(X_regimes[:split]),
        torch.tensor(y_targets[:split]),
    )
    ds_val = TensorDataset(
        torch.tensor(X_windows[split:]),
        torch.tensor(X_regimes[split:]),
        torch.tensor(y_targets[split:]),
    )
    loader_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True)
    loader_val = DataLoader(ds_val, batch_size=batch_size)

    model = CVAEModel().to(DEVICE)
    optim_ = torch.optim.Adam(model.parameters(), lr=lr)
    best_val, wait, best_state = float("inf"), 0, None

    for epoch in range(epochs):
        model.train()
        epoch_kl = 0.0
        n_batches = 0
        for xw, xr, yt in loader_train:
            xw, xr, yt = xw.to(DEVICE), xr.to(DEVICE), yt.to(DEVICE)
            recon, mu, logvar = model(xw, xr, yt)
            loss, kl_val = cvae_loss(recon, yt, mu, logvar, beta)
            optim_.zero_grad()
            loss.backward()
            optim_.step()
            epoch_kl += kl_val
            n_batches += 1

        mean_kl = epoch_kl / max(n_batches, 1)
        if mean_kl < 0.05 and epoch > 5:
            beta = beta * 0.5

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xw, xr, yt in loader_val:
                xw, xr, yt = xw.to(DEVICE), xr.to(DEVICE), yt.to(DEVICE)
                recon, mu, logvar = model(xw, xr, yt)
                loss, _ = cvae_loss(recon, yt, mu, logvar, beta)
                val_loss += loss.item()
        val_loss /= max(len(loader_val), 1)

        if val_loss < best_val:
            best_val, wait = val_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

    residuals = []
    with torch.no_grad():
        for xw, xr, yt in loader_val:
            xw, xr, yt = xw.to(DEVICE), xr.to(DEVICE), yt.to(DEVICE)
            regime_oh = F.one_hot(xr, N_REGIMES).float()
            z = torch.randn(xw.size(0), model.latent_dim, device=DEVICE)
            pred = model.decode(z, xw, regime_oh)
            residuals.append((yt - pred).cpu().numpy())
    residual_std = float(np.std(np.concatenate(residuals))) if residuals else 1e-4
    return model, residual_std


def cvae_generate_next(
    window: np.ndarray, regime: int, cvae_model: CVAEModel, residual_std: float
) -> float:
    cvae_model.eval()
    with torch.no_grad():
        xw = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        regime_oh = F.one_hot(
            torch.tensor([regime], device=DEVICE), N_REGIMES
        ).float()
        z = torch.randn(1, cvae_model.latent_dim, device=DEVICE)
        pred = cvae_model.decode(z, xw, regime_oh)
    noise = np.random.randn() * residual_std * 0.1
    return float(pred.cpu().item()) + noise

In [13]:
def cosine_beta_schedule(T: int) -> torch.Tensor:
    s = 0.008
    steps = torch.linspace(0, T, T + 1)
    alpha_bar = torch.cos(((steps / T) + s) / (1 + s) * (np.pi / 2)) ** 2
    alpha_bar = alpha_bar / alpha_bar[0]
    betas = 1 - alpha_bar[1:] / alpha_bar[:-1]
    return betas.clamp(0.0001, 0.999)


class FiLM(nn.Module):
    def __init__(self, cond_dim: int, feat_dim: int):
        super().__init__()
        self.gamma = nn.Linear(cond_dim, feat_dim)
        self.beta = nn.Linear(cond_dim, feat_dim)

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        return self.gamma(cond) * x + self.beta(cond)


class DiffusionDenoiser(nn.Module):
    def __init__(
        self,
        window: int = WINDOW,
        hidden_dim: int = 256,
        time_dim: int = 32,
        regime_dim: int = 16,
        T: int = 100,
    ):
        super().__init__()
        self.T = T

        self.ctx_conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.ctx_proj = nn.Linear(64, hidden_dim)

        self.time_embed = nn.Sequential(
            nn.Embedding(T, time_dim),
            nn.Linear(time_dim, hidden_dim),
            nn.SiLU(),
        )

        self.regime_embed = nn.Linear(N_REGIMES, regime_dim)
        self.film = FiLM(regime_dim, hidden_dim)

        self.net = nn.Sequential(
            nn.Linear(hidden_dim + 1, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(
        self,
        noisy_y: torch.Tensor,
        t: torch.Tensor,
        window: torch.Tensor,
        regime: torch.Tensor,
    ) -> torch.Tensor:

        ctx = self.ctx_conv(window.unsqueeze(1))  # (B, 64, 1)
        ctx = self.ctx_proj(ctx.squeeze(-1))

        t_emb = self.time_embed(t)

        h = ctx + t_emb

        reg_oh = F.one_hot(regime, N_REGIMES).float()
        reg_emb = self.regime_embed(reg_oh)
        h = self.film(h, reg_emb)

        h = torch.cat([h, noisy_y.unsqueeze(-1)], dim=-1)
        return self.net(h).squeeze(-1)


class ScoreDiffusionModel:
    def __init__(self, T_train: int = 100, T_infer: int = 20):
        self.T_train = T_train
        self.T_infer = T_infer
        betas = cosine_beta_schedule(T_train)
        self.alphas = 1.0 - betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)
        self.denoiser = DiffusionDenoiser(T=T_train).to(DEVICE)
        self.alpha_bar_dev = self.alpha_bar.to(DEVICE)

    def q_sample(
        self, y0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor
    ) -> torch.Tensor:
        ab = self.alpha_bar_dev[t]
        return torch.sqrt(ab) * y0 + torch.sqrt(1 - ab) * noise

    def train_step(
        self,
        y0: torch.Tensor,
        window: torch.Tensor,
        regime: torch.Tensor,
        optim_: torch.optim.Optimizer,
    ) -> float:
        t = torch.randint(0, self.T_train, (y0.size(0),), device=DEVICE)
        noise = torch.randn_like(y0)
        noisy_y = self.q_sample(y0, t, noise)
        pred_noise = self.denoiser(noisy_y, t, window, regime)
        loss = F.mse_loss(pred_noise, noise)
        optim_.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.denoiser.parameters(), 1.0)
        optim_.step()
        return loss.item()

    @torch.no_grad()
    def ddim_sample(
        self, window: torch.Tensor, regime: torch.Tensor, w: float = 2.0
    ) -> torch.Tensor:

        B = window.size(0)
        y = torch.randn(B, device=DEVICE)

        step_size = self.T_train // self.T_infer
        timesteps = list(range(self.T_train - 1, -1, -step_size))

        for i, t_val in enumerate(timesteps):
            t = torch.full((B,), t_val, device=DEVICE, dtype=torch.long)

            eps_cond = self.denoiser(y, t, window, regime)

            null_regime = torch.zeros(B, dtype=torch.long, device=DEVICE)
            eps_uncond = self.denoiser(y, t, window, null_regime)

            eps = (1 + w) * eps_cond - w * eps_uncond

            ab_t = self.alpha_bar_dev[t_val]
            pred_y0 = (y - torch.sqrt(1 - ab_t) * eps) / torch.sqrt(ab_t)

            if i < len(timesteps) - 1:
                t_prev = timesteps[i + 1]
                ab_prev = self.alpha_bar_dev[t_prev]
                y = torch.sqrt(ab_prev) * pred_y0 + torch.sqrt(1 - ab_prev) * eps
            else:
                y = pred_y0
        return y


def train_diffusion(
    X_windows: np.ndarray,
    X_regimes: np.ndarray,
    y_targets: np.ndarray,
    epochs: int = 60,
    batch_size: int = 256,
    lr: float = 1e-3,
    cfg_drop_prob: float = 0.1,
) -> Tuple[ScoreDiffusionModel, float]:

    n = len(y_targets)
    split = int(0.85 * n)
    ds_train = TensorDataset(
        torch.tensor(X_windows[:split]),
        torch.tensor(X_regimes[:split]),
        torch.tensor(y_targets[:split]),
    )
    ds_val = TensorDataset(
        torch.tensor(X_windows[split:]),
        torch.tensor(X_regimes[split:]),
        torch.tensor(y_targets[split:]),
    )
    loader_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True)
    loader_val = DataLoader(ds_val, batch_size=batch_size)

    diff = ScoreDiffusionModel()
    optim_ = torch.optim.Adam(diff.denoiser.parameters(), lr=lr)

    for epoch in range(epochs):
        diff.denoiser.train()
        for xw, xr, yt in loader_train:
            xw, xr, yt = xw.to(DEVICE), xr.to(DEVICE), yt.to(DEVICE)
            # Classifier-free guidance: randomly zero out regime
            drop_mask = torch.rand(xr.size(0), device=DEVICE) < cfg_drop_prob
            xr = xr * (~drop_mask).long()
            diff.train_step(yt, xw, xr, optim_)

    diff.denoiser.eval()

    residuals = []
    with torch.no_grad():
        for xw, xr, yt in loader_val:
            xw, xr, yt = xw.to(DEVICE), xr.to(DEVICE), yt.to(DEVICE)
            pred = diff.ddim_sample(xw, xr, w=2.0)
            residuals.append((yt - pred).cpu().numpy())
    residual_std = float(np.std(np.concatenate(residuals))) if residuals else 1e-4
    return diff, residual_std


def diffusion_generate_next(
    window: np.ndarray,
    regime: int,
    diff_model: ScoreDiffusionModel,
    residual_std: float,
) -> float:
    diff_model.denoiser.eval()
    with torch.no_grad():
        xw = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        xr = torch.tensor([regime], dtype=torch.long).to(DEVICE)
        sample = diff_model.ddim_sample(xw, xr, w=2.0)
    noise = np.random.randn() * residual_std * 0.1
    return float(sample.cpu().item()) + noise

In [14]:
def train_all() -> Dict:
    print("=== Fetching VIX + TNX data ===")
    vix_raw, features_df, X_scaled, scaler, dates = fetch_vix_pipeline()

    print("=== Fetching equities returns ===")
    returns_df = fetch_equities_returns()

    print("=== Fitting GMM (K=3) ===")
    gmm, regime_labels_raw = fit_regime_gmm(X_scaled, features_df, dates)
    regime_labels = smooth_regime_labels(regime_labels_raw)
    print(f"  Regime distribution:\n{regime_labels.value_counts(normalize=True).to_string()}")

    print("=== Estimating Markov chain ===")
    trans_matrix = estimate_transition_matrix(regime_labels)
    markov_step = make_markov_step(trans_matrix)
    print(f"  Transition matrix:\n{trans_matrix}")

    print("=== Building pooled training dataset ===")
    X_windows, X_regimes, y_targets = build_training_dataset(returns_df, regime_labels)
    print(f"  Dataset: {len(y_targets)} samples, window={WINDOW}")

    print("=== Training GBM ===")
    gbm_params = train_gbm(returns_df, regime_labels)

    print("=== Training GARCH(1,1) ===")
    garch_params = train_garch(returns_df, regime_labels)

    print("=== Calibrating Heston ===")
    heston_params = train_heston(returns_df, regime_labels)

    print("=== Training TFT ===")
    tft_model, tft_residual = train_tft(X_windows, X_regimes, y_targets)

    print("=== Training CVAE ===")
    cvae_model, cvae_residual = train_cvae(X_windows, X_regimes, y_targets)

    print("=== Training Diffusion ===")
    diff_model, diff_residual = train_diffusion(X_windows, X_regimes, y_targets)

    print("=== All training complete ===")
    return {
        "gmm": gmm,
        "scaler": scaler,
        "regime_labels": regime_labels,
        "transition_matrix": trans_matrix,
        "markov_step": markov_step,
        "training_data": (X_windows, X_regimes, y_targets),
        "gbm_params": gbm_params,
        "garch_params": garch_params,
        "heston_params": heston_params,
        "tft_model": tft_model,
        "tft_residual_std": tft_residual,
        "cvae_model": cvae_model,
        "cvae_residual_std": cvae_residual,
        "diffusion_model": diff_model,
        "diffusion_residual_std": diff_residual,
    }

In [15]:
def fetch_individual_returns(
    tickers: List[str] = TICKERS,
    start: str = "2005-01-01",
    end: str = "2025-12-31",
) -> Dict[str, pd.Series]:
    """Fetch per-equity daily log-returns as individual Series keyed by ticker."""
    result: Dict[str, pd.Series] = {}
    for t in tickers:
        prices = yf.download(t, start=start, end=end, progress=False)["Close"].squeeze()
        log_ret = np.log(prices / prices.shift(1)).iloc[1:].dropna()
        log_ret.name = t
        result[t] = log_ret
    return result

In [16]:
SEED_WINDOW = 60
N_TOTAL = 20_000
N_REAL = 10_000
N_REAL_POOLED = 5_000
N_REAL_INDIVIDUAL = 5_000
N_MODEL_SYNTHETIC = 5_000
N_FAKE = 5_000
N_FAKE_METHODS = 12
N_MODELS = 6


def _regime_to_int(labels: pd.Series) -> pd.Series:
    mapping = {name: i for i, name in enumerate(REGIME_NAMES)}
    return labels.map(mapping)


def _sample_real_window(
    returns_series: pd.Series,
    regime_labels_int: pd.Series,
    length: int = SEQUENCE_LENGTH,
) -> Optional[Tuple[np.ndarray, np.ndarray]]:

    common = returns_series.index.intersection(regime_labels_int.index)
    if len(common) < length:
        return None
    ret_aligned = returns_series.loc[common].values
    reg_aligned = regime_labels_int.loc[common].values
    max_start = len(ret_aligned) - length
    if max_start < 1:
        return None
    start = np.random.randint(0, max_start)
    r = ret_aligned[start : start + length].astype(np.float32)
    g = reg_aligned[start : start + length]
    if np.any(np.isnan(r)) or np.any(np.isnan(g)):
        return None
    return r, g.astype(np.float32)


def _sample_real_sequences(
    source_series: List[pd.Series],
    regime_labels_int: pd.Series,
    n: int,
) -> Tuple[np.ndarray, np.ndarray]:

    returns_out = np.empty((n, SEQUENCE_LENGTH), dtype=np.float32)
    regimes_out = np.empty((n, SEQUENCE_LENGTH), dtype=np.float32)
    collected = 0
    while collected < n:
        series = source_series[np.random.randint(len(source_series))]
        result = _sample_real_window(series, regime_labels_int)
        if result is not None:
            returns_out[collected], regimes_out[collected] = result
            collected += 1
    return returns_out, regimes_out


def _autoregressive_generate(
    seed: np.ndarray,
    regime_sequence: np.ndarray,
    gen_fn: Callable[[np.ndarray, int], float],
    window: int = WINDOW,
) -> np.ndarray:

    buffer = list(seed)
    for t in range(SEQUENCE_LENGTH):
        w = np.array(buffer[-window:], dtype=np.float32)
        r = gen_fn(w, int(regime_sequence[t]))
        if not np.isfinite(r):
            r = 0.0
        buffer.append(float(r))
    return np.array(buffer[-SEQUENCE_LENGTH:], dtype=np.float32)


def _distribute_counts(total: int, n_buckets: int) -> List[int]:

    base = total // n_buckets
    remainder = total % n_buckets
    return [base + (1 if i < remainder else 0) for i in range(n_buckets)]


def _build_model_synthetic(
    train_all_dict: Dict,
    individual_returns: Dict[str, pd.Series],
    regime_labels: pd.Series,
    n: int = N_MODEL_SYNTHETIC,
) -> Tuple[np.ndarray, np.ndarray]:

    trans_matrix = train_all_dict["transition_matrix"]
    gbm_params = train_all_dict["gbm_params"]
    garch_params = train_all_dict["garch_params"]
    heston_params = train_all_dict["heston_params"]
    tft_model = train_all_dict["tft_model"]
    tft_res = train_all_dict["tft_residual_std"]
    cvae_model = train_all_dict["cvae_model"]
    cvae_res = train_all_dict["cvae_residual_std"]
    diff_model = train_all_dict["diffusion_model"]
    diff_res = train_all_dict["diffusion_residual_std"]

    generators: List[Tuple[str, Callable[[np.ndarray, int], float]]] = [
        ("gbm", lambda w, r: gbm_generate_next(w, r, gbm_params)),
        ("garch", lambda w, r: garch_generate_next(w, r, garch_params)),
        ("heston", lambda w, r: heston_generate_next(w, r, heston_params)),
        ("tft", lambda w, r: tft_generate_next(w, r, tft_model, tft_res)),
        ("cvae", lambda w, r: cvae_generate_next(w, r, cvae_model, cvae_res)),
        ("diffusion", lambda w, r: diffusion_generate_next(w, r, diff_model, diff_res)),
    ]

    regime_to_idx = {name: i for i, name in enumerate(REGIME_NAMES)}
    regime_labels_int = regime_labels.map(regime_to_idx).dropna().astype(int)

    valid_tickers = []
    for t, s in individual_returns.items():
        common = s.index.intersection(regime_labels_int.index)
        if len(common) >= SEED_WINDOW + 1:
            valid_tickers.append(t)

    per_model = per_model_map = {
        "gbm": 1000,
        "garch": 1000,
        "heston": 1000,
        "tft": 900,
        "cvae": 1000,
        "diffusion": 100,
}

    returns_out = np.empty((n, SEQUENCE_LENGTH), dtype=np.float32)
    regimes_out = np.empty((n, SEQUENCE_LENGTH), dtype=np.float32)
    idx = 0

    for name, gen_fn in generators:
        count = per_model_map[name]
        print(f"  Generating {count} sequences with {name}...")
        generated = 0
        while generated < count:
            ticker = valid_tickers[np.random.randint(len(valid_tickers))]
            series = individual_returns[ticker]
            common_dates = series.index.intersection(regime_labels_int.index)
            series_aligned = series.loc[common_dates]
            reg_aligned = regime_labels_int.loc[common_dates]

            if len(series_aligned) < SEED_WINDOW + 1:
                continue
            start = np.random.randint(0, len(series_aligned) - SEED_WINDOW)
            seed = series_aligned.values[start : start + SEED_WINDOW].astype(np.float32)
            if np.any(~np.isfinite(seed)):
                continue

            initial_regime = int(reg_aligned.values[start + SEED_WINDOW - 1])
            regime_seq = generate_regime_sequence(trans_matrix, initial_regime, SEQUENCE_LENGTH)
            ret_seq = _autoregressive_generate(seed, regime_seq, gen_fn)

            if np.any(~np.isfinite(ret_seq)):
                continue

            returns_out[idx] = ret_seq
            regimes_out[idx] = regime_seq.astype(np.float32)
            idx += 1
            generated += 1

    return returns_out, regimes_out


def _fake_constant_drift(n: int = SEQUENCE_LENGTH) -> np.ndarray:
    drift = np.random.uniform(0.0005, 0.005)
    sigma = np.random.uniform(0.00005, 0.0005)
    return (np.full(n, drift) + np.random.randn(n) * sigma).astype(np.float32)


def _fake_iid_gaussian(n: int = SEQUENCE_LENGTH) -> np.ndarray:
    sigma = np.random.uniform(0.005, 0.03)
    return (np.random.randn(n) * sigma).astype(np.float32)


def _fake_perfect_alternation(n: int = SEQUENCE_LENGTH) -> np.ndarray:
    amplitude = np.random.uniform(0.003, 0.02)
    signs = np.array([(-1) ** i for i in range(n)], dtype=np.float64)
    if np.random.rand() < 0.3:
        for i in range(1, n - 1):
            if np.random.rand() < 0.15:
                signs[i] = signs[i + 1] if np.random.rand() < 0.5 else signs[i - 1]
    return (signs * amplitude).astype(np.float32)


def _fake_linear_trend(n: int = SEQUENCE_LENGTH) -> np.ndarray:
    start_val = np.random.uniform(-0.01, -0.001)
    end_val = np.random.uniform(0.001, 0.01)
    return np.linspace(start_val, end_val, n).astype(np.float32)


def _fake_sinusoidal(n: int = SEQUENCE_LENGTH) -> np.ndarray:
    amplitude = np.random.uniform(0.005, 0.02)
    period = np.random.uniform(10, 40)
    t = np.arange(n)
    return (amplitude * np.sin(2 * np.pi * t / period)).astype(np.float32)


def _fake_non_negative(n: int = SEQUENCE_LENGTH) -> np.ndarray:
    sigma = np.random.uniform(0.005, 0.03)
    return np.abs(np.random.randn(n) * sigma).astype(np.float32)


def _fake_frequent_jumps(n: int = SEQUENCE_LENGTH) -> np.ndarray:
    magnitude = np.random.uniform(0.05, 0.15)
    freq = int(np.random.uniform(3, 10))
    seq = np.zeros(n, dtype=np.float32)
    for i in range(0, n, max(freq, 1)):
        seq[i] = magnitude * np.random.choice([-1.0, 1.0])
    return seq


def _fake_piecewise_deterministic(n: int = SEQUENCE_LENGTH) -> np.ndarray:
    boundaries = sorted(np.random.choice(range(1, n - 1), size=2, replace=False))
    segments = [0] + list(boundaries) + [n]
    seq = np.empty(n, dtype=np.float32)
    for i in range(3):
        val = np.random.uniform(-0.005, 0.005)
        seq[segments[i] : segments[i + 1]] = val
    return seq


def _fake_quantized(n: int = SEQUENCE_LENGTH) -> np.ndarray:
    n_levels = np.random.randint(3, 6)
    levels = np.random.uniform(-0.02, 0.02, size=n_levels).astype(np.float32)
    return np.random.choice(levels, size=n).astype(np.float32)


def _fake_exploding_variance(n: int = SEQUENCE_LENGTH) -> np.ndarray:
    base_sigma = np.random.uniform(0.0005, 0.002)
    growth = np.random.uniform(0.00005, 0.0002)
    sigmas = base_sigma + growth * np.arange(n)
    return (np.random.randn(n) * sigmas).astype(np.float32)


def _fake_mechanical_mean_reversion(n: int = SEQUENCE_LENGTH) -> np.ndarray:
    coeff = np.random.uniform(-0.99, -0.80)
    x = np.random.uniform(-0.01, 0.01)
    seq = np.empty(n, dtype=np.float32)
    for i in range(n):
        seq[i] = x
        x = coeff * x
    return seq


def _fake_copy_pasted_blocks(n: int = SEQUENCE_LENGTH) -> np.ndarray:
    block_len = np.random.randint(20, 81)
    block = (np.random.randn(block_len) * 0.01).astype(np.float32)
    n_repeats = int(np.ceil(n / block_len))
    tiled = np.tile(block, n_repeats)
    return tiled[:n]


FAKE_GENERATORS = [
    _fake_constant_drift,
    _fake_iid_gaussian,
    _fake_perfect_alternation,
    _fake_linear_trend,
    _fake_sinusoidal,
    _fake_non_negative,
    _fake_frequent_jumps,
    _fake_piecewise_deterministic,
    _fake_quantized,
    _fake_exploding_variance,
    _fake_mechanical_mean_reversion,
    _fake_copy_pasted_blocks,
]


def _build_fake_synthetic(
    trans_matrix: np.ndarray, n: int = N_FAKE
) -> Tuple[np.ndarray, np.ndarray]:
    per_method = _distribute_counts(n, N_FAKE_METHODS)
    returns_out = np.empty((n, SEQUENCE_LENGTH), dtype=np.float32)
    regimes_out = np.empty((n, SEQUENCE_LENGTH), dtype=np.float32)
    idx = 0
    for gen_fn, count in zip(FAKE_GENERATORS, per_method):
        for _ in range(count):
            returns_out[idx] = gen_fn(SEQUENCE_LENGTH)
            initial_regime = np.random.randint(N_REGIMES)
            regimes_out[idx] = generate_regime_sequence(
                trans_matrix, initial_regime, SEQUENCE_LENGTH
            ).astype(np.float32)
            idx += 1
    return returns_out, regimes_out


def build_discriminator_dataset(
    train_all_dict: Dict,
    individual_returns: Dict[str, pd.Series],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:

    regime_labels = train_all_dict["regime_labels"]
    trans_matrix = train_all_dict["transition_matrix"]
    regime_to_idx = {name: i for i, name in enumerate(REGIME_NAMES)}
    regime_labels_int = regime_labels.map(regime_to_idx).dropna().astype(int)

    print("=== Sampling 5,000 real sequences (pooled) ===")
    pooled_df = pd.DataFrame(individual_returns).dropna()
    pooled_series_list = [pooled_df[col] for col in pooled_df.columns]
    real_pooled_r, real_pooled_g = _sample_real_sequences(
        pooled_series_list, regime_labels_int, N_REAL_POOLED
    )

    print("=== Sampling 5,000 real sequences (individual) ===")
    individual_series_list = list(individual_returns.values())
    real_indiv_r, real_indiv_g = _sample_real_sequences(
        individual_series_list, regime_labels_int, N_REAL_INDIVIDUAL
    )

    print("=== Generating 5,000 model-based synthetic sequences ===")
    model_r, model_g = _build_model_synthetic(
        train_all_dict, individual_returns, regime_labels, N_MODEL_SYNTHETIC
    )

    print("=== Generating 5,000 obviously-fake sequences ===")
    fake_r, fake_g = _build_fake_synthetic(trans_matrix, N_FAKE)

    X_returns = np.concatenate([real_pooled_r, real_indiv_r, model_r, fake_r], axis=0)
    X_regimes = np.concatenate([real_pooled_g, real_indiv_g, model_g, fake_g], axis=0)
    y = np.concatenate([
        np.zeros(N_REAL, dtype=np.int64),
        np.ones(N_MODEL_SYNTHETIC + N_FAKE, dtype=np.int64),
    ])

    perm = np.random.permutation(N_TOTAL)
    X_returns, X_regimes, y = X_returns[perm], X_regimes[perm], y[perm]

    print(f"=== Discriminator dataset: {X_returns.shape[0]} instances, "
          f"class balance: {y.mean():.2%} synthetic ===")
    return X_returns, X_regimes, y

In [22]:
class CausalConv1d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, dilation: int):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(F.pad(x, (self.pad, 0)))


class TCNBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, dilation: int):
        super().__init__()
        self.conv1 = CausalConv1d(in_ch, out_ch, kernel_size, dilation)
        self.norm1 = nn.GroupNorm(1, out_ch)
        self.conv2 = CausalConv1d(out_ch, out_ch, kernel_size, dilation)
        self.norm2 = nn.GroupNorm(1, out_ch)
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res = self.residual(x)
        h = F.relu(self.norm1(self.conv1(x)))
        h = self.dropout(h)
        h = F.relu(self.norm2(self.conv2(h)))
        h = self.dropout(h)
        return h + res


class TCNDiscriminator(nn.Module):
    DILATIONS = [1, 2, 4, 8, 16, 32]
    KERNEL_SIZE = 3

    def __init__(self, in_channels: int = 2, hidden_channels: int = 64):
        super().__init__()
        blocks: List[nn.Module] = []
        ch_in = in_channels
        for d in self.DILATIONS:
            blocks.append(TCNBlock(ch_in, hidden_channels, self.KERNEL_SIZE, d))
            ch_in = hidden_channels
        self.tcn = nn.Sequential(*blocks)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(hidden_channels, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 1)  # (B, 252, 2) → (B, 2, 252)
        x = self.tcn(x)
        return self.head(x).squeeze(-1)


def build_tcn(sequence_length: int = SEQUENCE_LENGTH) -> TCNDiscriminator:
    return TCNDiscriminator(in_channels=2, hidden_channels=64)


def _compute_accuracy(model: TCNDiscriminator, loader: DataLoader) -> float:
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds = (torch.sigmoid(model(xb)) >= 0.5).long()
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / max(total, 1)


def train_tcn(
    X_returns: np.ndarray,
    X_regimes: np.ndarray,
    y: np.ndarray,
    n_ensemble: int = 5,
    epochs: int = 30,
    batch_size: int = 256,
    lr: float = 1e-3,
    patience: int = 5,
    train_frac: float = 0.80,
    val_frac: float = 0.10,
) -> Dict:

    from sklearn.linear_model import LogisticRegression

    X = np.stack([X_returns, X_regimes], axis=-1).astype(np.float32)
    n = len(y)

    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    X_train, y_train = X[:n_train], y[:n_train]
    X_val, y_val = X[n_train : n_train + n_val], y[n_train : n_train + n_val]
    X_cal, y_cal = X[n_train + n_val :], y[n_train + n_val :]

    ds_train = TensorDataset(
        torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long)
    )
    ds_val = TensorDataset(
        torch.tensor(X_val), torch.tensor(y_val, dtype=torch.long)
    )
    ds_cal = TensorDataset(
        torch.tensor(X_cal), torch.tensor(y_cal, dtype=torch.long)
    )
    loader_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True)
    loader_val = DataLoader(ds_val, batch_size=batch_size)
    loader_cal = DataLoader(ds_cal, batch_size=batch_size)

    models: List[TCNDiscriminator] = []
    memorisation_flags: List[bool] = []

    for seed_idx in range(n_ensemble):
        print(f"  Training TCN ensemble member {seed_idx + 1}/{n_ensemble}...")
        torch.manual_seed(seed_idx * 42)
        np.random.seed(seed_idx * 42)

        model = build_tcn().to(DEVICE)
        optim_ = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optim_, patience=3, factor=0.5
        )
        best_val_loss, wait, best_state = float("inf"), 0, None

        for epoch in range(epochs):
            model.train()
            epoch_loss, n_batches = 0.0, 0
            for xb, yb in loader_train:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE).float()
                logits = model(xb)
                loss = F.binary_cross_entropy_with_logits(logits, yb)
                optim_.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optim_.step()
                epoch_loss += loss.item()
                n_batches += 1

            model.eval()
            val_loss, n_vb = 0.0, 0
            with torch.no_grad():
                for xb, yb in loader_val:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE).float()
                    val_loss += F.binary_cross_entropy_with_logits(model(xb), yb).item()
                    n_vb += 1
            val_loss /= max(n_vb, 1)
            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss, wait = val_loss, 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= patience:
                    print(f"    Early stop at epoch {epoch + 1}")
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        model.eval()

        train_acc = _compute_accuracy(model, loader_train)
        val_acc = _compute_accuracy(model, loader_val)
        memorised = train_acc >= 0.90 and val_acc <= 0.60
        memorisation_flags.append(memorised)
        print(f"    train_acc={train_acc:.3f}, val_acc={val_acc:.3f}, "
              f"memorised={'YES — will not deploy' if memorised else 'no'}")
        models.append(model)

    deployed = [m for m, flag in zip(models, memorisation_flags) if not flag]
    if len(deployed) == 0:
        print("  WARNING: All ensemble members memorised. Deploying all with caution.")
        deployed = models

    print("  Platt scaling on calibration set...")
    all_probs_cal: List[np.ndarray] = []
    all_labels_cal: List[np.ndarray] = []
    for xb, yb in loader_cal:
        xb = xb.to(DEVICE)
        member_probs = []
        with torch.no_grad():
            for m in deployed:
                m.eval()
                member_probs.append(torch.sigmoid(m(xb)).cpu().numpy())
        all_probs_cal.append(np.mean(member_probs, axis=0))
        all_labels_cal.append(yb.numpy())

    cal_probs = np.concatenate(all_probs_cal)
    cal_labels = np.concatenate(all_labels_cal)

    platt = LogisticRegression(C=1e10, solver="lbfgs", max_iter=1000)
    platt.fit(cal_probs.reshape(-1, 1), cal_labels)

    cal_acc = ((cal_probs >= 0.5).astype(int) == cal_labels).mean()
    print(f"  Calibration accuracy (pre-Platt): {cal_acc:.3f}")
    print(f"  Deployed {len(deployed)}/{n_ensemble} ensemble members")

    return {
        "models": deployed,
        "platt_scaler": platt,
        "memorisation_flags": memorisation_flags,
        "n_deployed": len(deployed),
    }


def predict_tcn(
    X_returns: np.ndarray,
    X_regimes: np.ndarray,
    tcn_dict: Dict,
    batch_size: int = 512,
) -> Tuple[np.ndarray, np.ndarray]:

    models = tcn_dict["models"]
    platt = tcn_dict["platt_scaler"]
    X = np.stack([X_returns, X_regimes], axis=-1).astype(np.float32)
    ds = TensorDataset(torch.tensor(X))
    loader = DataLoader(ds, batch_size=batch_size)

    all_probs: List[np.ndarray] = []
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        member_out = []
        with torch.no_grad():
            for m in models:
                m.eval()
                member_out.append(torch.sigmoid(m(xb)).cpu().numpy())
        all_probs.append(np.mean(member_out, axis=0))

    raw_probs = np.concatenate(all_probs)
    calibrated = platt.predict_proba(raw_probs.reshape(-1, 1))[:, 1]
    labels = (calibrated >= 0.5).astype(int)
    return labels, calibrated

In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

def main():
    np.random.seed(42)
    torch.manual_seed(42)

    print("=== Stage 1: train all generators / regime machinery ===")
    train_all_dict = train_all()

    print("=== Stage 2: fetch individual returns ===")
    individual_returns = fetch_individual_returns()

    print("=== Stage 3: build discriminator dataset ===")
    X_returns, X_regimes, y = build_discriminator_dataset(
        train_all_dict=train_all_dict,
        individual_returns=individual_returns,
    )

    print(f"X_returns shape: {X_returns.shape}")
    print(f"X_regimes shape: {X_regimes.shape}")
    print(f"y shape: {y.shape}")
    print(f"Synthetic fraction: {y.mean():.2%}")

    n = len(y)
    test_frac = 0.20
    n_test = int(test_frac * n)

    X_returns_train = X_returns[:-n_test]
    X_regimes_train = X_regimes[:-n_test]
    y_train = y[:-n_test]

    X_returns_test = X_returns[-n_test:]
    X_regimes_test = X_regimes[-n_test:]
    y_test = y[-n_test:]

    print("=== Stage 4: train TCN on train split ===")
    tcn_dict = train_tcn(
        X_returns=X_returns_train,
        X_regimes=X_regimes_train,
        y=y_train,
        n_ensemble=5,
        epochs=30,
        batch_size=256,
        lr=1e-3,
        patience=5,
        train_frac=0.80,
        val_frac=0.10,
    )

    print("=== Stage 5: test TCN on held-out split ===")
    y_pred, y_prob = predict_tcn(
        X_returns=X_returns_test,
        X_regimes=X_regimes_test,
        tcn_dict=tcn_dict,
        batch_size=512,
    )

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_test, y_prob)
    except ValueError:
        auc = float("nan")

    cm = confusion_matrix(y_test, y_pred)

    print("\n=== HELD-OUT TEST RESULTS ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 score : {f1:.4f}")
    print(f"ROC AUC  : {auc:.4f}")
    print("\nConfusion matrix:")
    print(cm)

    print("\nClassification report:")
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))

    return {
        "train_all_dict": train_all_dict,
        "individual_returns": individual_returns,
        "X_returns_train": X_returns_train,
        "X_regimes_train": X_regimes_train,
        "y_train": y_train,
        "X_returns_test": X_returns_test,
        "X_regimes_test": X_regimes_test,
        "y_test": y_test,
        "tcn_dict": tcn_dict,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "metrics": {
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "roc_auc": auc,
            "confusion_matrix": cm,
        },
    }


if __name__ == "__main__":
    results = main()

=== Stage 1: train all generators / regime machinery ===
=== Fetching VIX + TNX data ===


/tmp/ipykernel_623/3695538259.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  vix = yf.download("^VIX", start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/3695538259.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  tnx = yf.download("^TNX", start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/622082558.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(tickers, start=start, end=end, progress=False)["Close"]


=== Fetching equities returns ===
=== Fitting GMM (K=3) ===
  Regime distribution:
regime
low_vol     0.520814
mid_vol     0.416271
high_vol    0.062916
=== Estimating Markov chain ===
  Transition matrix:
[[0.98028478 0.01861993 0.00109529]
 [0.02465753 0.9716895  0.00365297]
 [0.         0.03323263 0.96676737]]
=== Building pooled training dataset ===
  Dataset: 78450 samples, window=30
=== Training GBM ===
=== Training GARCH(1,1) ===
=== Calibrating Heston ===
=== Training TFT ===
=== Training CVAE ===
=== Training Diffusion ===
=== All training complete ===
=== Stage 2: fetch individual returns ===


/tmp/ipykernel_623/86867600.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(t, start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/86867600.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(t, start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/86867600.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(t, start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/86867600.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(t, start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/86867600.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(t, start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/868676

=== Stage 3: build discriminator dataset ===
=== Sampling 5,000 real sequences (pooled) ===
=== Sampling 5,000 real sequences (individual) ===
=== Generating 5,000 model-based synthetic sequences ===
  Generating 1000 sequences with gbm...
  Generating 1000 sequences with garch...
  Generating 1000 sequences with heston...
  Generating 900 sequences with tft...
  Generating 1000 sequences with cvae...
  Generating 100 sequences with diffusion...
=== Generating 5,000 obviously-fake sequences ===
=== Discriminator dataset: 20000 instances, class balance: 50.00% synthetic ===
X_returns shape: (20000, 252)
X_regimes shape: (20000, 252)
y shape: (20000,)
Synthetic fraction: 50.00%
=== Stage 4: train TCN on train split ===
  Training TCN ensemble member 1/5...
    train_acc=0.982, val_acc=0.979, memorised=no
  Training TCN ensemble member 2/5...
    Early stop at epoch 16
    train_acc=0.899, val_acc=0.893, memorised=no
  Training TCN ensemble member 3/5...
    Early stop at epoch 16
    tra

In [20]:
import warnings
from scipy import stats as sp_stats
from scipy.optimize import curve_fit
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

def _hill_alpha(x: np.ndarray) -> Tuple[float, bool]:
    x = x[np.isfinite(x)]
    x = np.abs(x[x != 0])
    n = len(x)
    k = int(np.floor(n ** 0.6))
    if k < 4:
        return np.nan, True
    x_sorted = np.sort(x)[::-1]  # descending
    log_ratios = np.log(x_sorted[:k] / x_sorted[k])
    H_k = np.mean(log_ratios)
    if H_k <= 0:
        return np.nan, True
    return 1.0 / H_k, False


def _leverage_stat(returns: np.ndarray, max_tau: int = 10) -> float:
    returns = returns[np.isfinite(returns)]
    L_plus = 0.0
    for tau in range(1, max_tau + 1):
        if len(returns) <= tau + 1:
            break
        r_t = returns[: len(returns) - tau]
        r2_t_tau = returns[tau:] ** 2
        if np.std(r_t) < 1e-12 or np.std(r2_t_tau) < 1e-12:
            continue
        corr = np.corrcoef(r_t, r2_t_tau)[0, 1]
        if np.isfinite(corr):
            L_plus += corr
    return L_plus


def _leverage_bootstrap_sigma(
    returns: np.ndarray, n_boot: int = 500, max_tau: int = 10
) -> float:
    L_plus_samples = []
    for _ in range(n_boot):
        shuffled = np.random.permutation(returns)
        L_plus_samples.append(_leverage_stat(shuffled, max_tau))
    return float(np.std(L_plus_samples))


def _acf_array(x: np.ndarray, max_lag: int = 50) -> np.ndarray:
    x = x - np.mean(x)
    n = len(x)
    var = np.var(x)
    if var < 1e-16:
        return np.zeros(max_lag)
    usable_lags = min(max_lag, n - 2)
    acf = np.zeros(max_lag)
    for lag in range(1, usable_lags + 1):
        acf[lag - 1] = np.mean(x[: n - lag] * x[lag:]) / var
    return acf


def _power_law(tau: np.ndarray, a: float, beta: float) -> np.ndarray:
    return a * tau ** (-beta)


def _exponential(tau: np.ndarray, a: float, b: float) -> np.ndarray:
    return a * np.exp(-b * tau)


def compute_benchmark(
    individual_returns: Dict[str, pd.Series],
    regime_labels: pd.Series,
) -> Dict[str, Dict]:

    regime_to_idx = {name: i for i, name in enumerate(REGIME_NAMES)}
    regime_labels_int = regime_labels.map(regime_to_idx).dropna().astype(int)

    benchmark: Dict[str, Dict] = {}

    for regime_name in REGIME_NAMES:
        regime_idx = regime_to_idx[regime_name]

        hill_alphas_upper: List[float] = []
        hill_alphas_lower: List[float] = []
        leverage_stats: List[float] = []

        for ticker, series in individual_returns.items():
            common = series.index.intersection(regime_labels_int.index)
            if len(common) < 30:
                continue
            ret = series.loc[common].values
            reg = regime_labels_int.loc[common].values
            mask = reg == regime_idx
            regime_ret = ret[mask]
            regime_ret = regime_ret[np.isfinite(regime_ret)]
            if len(regime_ret) < 30:
                continue

            pos = regime_ret[regime_ret > 0]
            if len(pos) >= 10:
                alpha, caution = _hill_alpha(pos)
                if not caution and np.isfinite(alpha):
                    hill_alphas_upper.append(alpha)

            neg = np.abs(regime_ret[regime_ret < 0])
            if len(neg) >= 10:
                alpha, caution = _hill_alpha(neg)
                if not caution and np.isfinite(alpha):
                    hill_alphas_lower.append(alpha)

            L_plus = _leverage_stat(regime_ret)
            if np.isfinite(L_plus):
                leverage_stats.append(L_plus)

        hill_combined = []
        for u, l in zip(hill_alphas_upper, hill_alphas_lower):
            hill_combined.append((u + l) / 2.0)
        if not hill_combined:
            hill_combined = [3.5]

        all_regime_ret = []
        for ticker, series in individual_returns.items():
            common = series.index.intersection(regime_labels_int.index)
            if len(common) < 30:
                continue
            ret = series.loc[common].values
            reg = regime_labels_int.loc[common].values
            regime_ret = ret[reg == regime_idx]
            regime_ret = regime_ret[np.isfinite(regime_ret)]
            all_regime_ret.extend(regime_ret.tolist())
        all_regime_ret_arr = np.array(all_regime_ret, dtype=np.float64)

        L_plus_real = _leverage_stat(all_regime_ret_arr)

        subsample_size = min(len(all_regime_ret_arr), 5000)
        boot_sample = np.random.choice(
            all_regime_ret_arr, size=subsample_size, replace=False
        )
        L_plus_sigma = _leverage_bootstrap_sigma(boot_sample, n_boot=500)

        benchmark[regime_name] = {
            "hill_alpha_mean": float(np.mean(hill_combined)),
            "hill_alpha_std": float(np.std(hill_combined)) if len(hill_combined) > 1 else 1.0,
            "leverage_L_plus": float(L_plus_real),
            "leverage_sigma": float(L_plus_sigma) if L_plus_sigma > 0 else 0.1,
        }

    return benchmark


def _test_hill(
    returns: np.ndarray, benchmark_regime: Dict
) -> Dict:
    """Hill tail index test (test_id: 'hill')."""
    pos = returns[returns > 0]
    neg = np.abs(returns[returns < 0])

    alpha_upper, caution_upper = _hill_alpha(pos) if len(pos) >= 5 else (np.nan, True)
    alpha_lower, caution_lower = _hill_alpha(neg) if len(neg) >= 5 else (np.nan, True)

    caution = caution_upper or caution_lower
    if np.isfinite(alpha_upper) and np.isfinite(alpha_lower):
        alpha_syn = (alpha_upper + alpha_lower) / 2.0
    elif np.isfinite(alpha_upper):
        alpha_syn = alpha_upper
    elif np.isfinite(alpha_lower):
        alpha_syn = alpha_lower
    else:
        return {
            "test_id": "hill_tail_index", "verdict": "fail",
            "polarity": "benchmark_comparison", "statistic": None,
            "p_value": None, "p_value_bh": None,
            "benchmark": benchmark_regime["hill_alpha_mean"],
            "auxiliary": {"caution": True},
        }

    alpha_real = benchmark_regime["hill_alpha_mean"]
    sigma = benchmark_regime["hill_alpha_std"]
    sigma = max(sigma, 0.1)  # floor to prevent division by near-zero

    diff = abs(alpha_syn - alpha_real)
    if diff <= 1.0 * sigma:
        verdict = "pass"
    elif diff <= 2.0 * sigma:
        verdict = "marginal"
    else:
        verdict = "fail"

    return {
        "test_id": "hill_tail_index", "verdict": verdict,
        "polarity": "benchmark_comparison",
        "statistic": float(alpha_syn), "p_value": None, "p_value_bh": None,
        "benchmark": float(alpha_real),
        "auxiliary": {
            "alpha_upper": float(alpha_upper) if np.isfinite(alpha_upper) else None,
            "alpha_lower": float(alpha_lower) if np.isfinite(alpha_lower) else None,
            "diff_sigma": float(diff / sigma),
            "caution": caution,
        },
    }


def _test_jarque_bera(returns: np.ndarray) -> Dict:
    jb_stat, p_value = sp_stats.jarque_bera(returns)

    if p_value < 0.05:
        verdict = "pass"
    elif p_value < 0.20:
        verdict = "marginal"
    else:
        verdict = "fail"

    return {
        "test_id": "jarque_bera", "verdict": verdict,
        "polarity": "should_reject",
        "statistic": float(jb_stat), "p_value": float(p_value), "p_value_bh": None,
        "benchmark": None,
        "auxiliary": {
            "skewness": float(sp_stats.skew(returns)),
            "kurtosis": float(sp_stats.kurtosis(returns)),
        },
    }


def _test_lb_raw(returns: np.ndarray) -> Dict:
    """Ljung-Box on raw returns, h=10 (test_id: 'lb_raw'). Polarity: should_not_reject."""
    demeaned = returns - np.mean(returns)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = acorr_ljungbox(demeaned, lags=10, return_df=True)
    p_value = float(result["lb_pvalue"].iloc[-1])
    lb_stat = float(result["lb_stat"].iloc[-1])

    if p_value > 0.05:
        verdict = "pass"
    elif p_value >= 0.01:
        verdict = "marginal"
    else:
        verdict = "fail"

    return {
        "test_id": "ljung_box_raw", "verdict": verdict,
        "polarity": "should_not_reject",
        "statistic": float(lb_stat), "p_value": float(p_value), "p_value_bh": None,
        "benchmark": None, "auxiliary": {},
    }


def _test_lb_squared(returns: np.ndarray) -> Dict:
    squared = returns ** 2
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = acorr_ljungbox(squared, lags=20, return_df=True)
    p_value = float(result["lb_pvalue"].iloc[-1])
    lb_stat = float(result["lb_stat"].iloc[-1])

    if p_value < 0.05:
        verdict = "pass"
    elif p_value < 0.20:
        verdict = "marginal"
    else:
        verdict = "fail"

    return {
        "test_id": "ljung_box_squared", "verdict": verdict,
        "polarity": "should_reject",
        "statistic": float(lb_stat), "p_value": float(p_value), "p_value_bh": None,
        "benchmark": None, "auxiliary": {},
    }


def _test_arch_lm(returns: np.ndarray) -> Dict:
    demeaned = returns - np.mean(returns)
    lm_stat, lm_p, f_stat, f_p = het_arch(demeaned, nlags=5)
    p_value = float(lm_p)

    if p_value < 0.05:
        verdict = "pass"
    elif p_value < 0.20:
        verdict = "marginal"
    else:
        verdict = "fail"

    return {
        "test_id": "arch_lm", "verdict": verdict,
        "polarity": "should_reject",
        "statistic": float(lm_stat), "p_value": float(p_value), "p_value_bh": None,
        "benchmark": None,
        "auxiliary": {"f_stat": float(f_stat), "f_pvalue": float(f_p)},
    }


def _test_leverage(returns: np.ndarray, benchmark_regime: Dict) -> Dict:
    L_plus_syn = _leverage_stat(returns)

    n_boot = 500
    boot_L = np.array([
        _leverage_stat(np.random.permutation(returns)) for _ in range(n_boot)
    ])
    boot_successful = np.sum(np.isfinite(boot_L))
    if boot_successful < n_boot:
        boot_L = boot_L[np.isfinite(boot_L)]

    if len(boot_L) > 0:
        p_value = float(np.mean(boot_L <= L_plus_syn))
    else:
        p_value = 1.0

    L_plus_real = benchmark_regime["leverage_L_plus"]
    sigma = benchmark_regime["leverage_sigma"]
    sigma = max(sigma, 1e-4)

    if L_plus_syn < L_plus_real - 0.5 * sigma:
        verdict = "pass"
    elif L_plus_syn <= 0:
        verdict = "marginal"
    else:
        verdict = "fail"

    return {
        "test_id": "leverage", "verdict": verdict,
        "polarity": "benchmark_comparison",
        "statistic": float(L_plus_syn), "p_value": float(p_value), "p_value_bh": None,
        "benchmark": float(L_plus_real),
        "auxiliary": {
            "sigma": float(sigma),
            "bootstrap_n": int(len(boot_L)),
        },
    }


def _test_acf_abs(returns: np.ndarray) -> Dict:
    abs_ret = np.abs(returns)
    acf = _acf_array(abs_ret, max_lag=50)
    lags = np.arange(1, 51, dtype=np.float64)

    n = len(returns)
    sig_threshold = 2.0 / np.sqrt(n)
    has_structure_beyond_5 = np.any(np.abs(acf[5:]) > sig_threshold)

    power_law_ok = False
    beta_hat = np.nan
    aic_pow = np.inf
    try:
        positive_mask = acf > 0
        if np.sum(positive_mask) >= 5:
            lags_pos = lags[positive_mask]
            acf_pos = acf[positive_mask]
            popt_pow, _ = curve_fit(
                _power_law, lags_pos, acf_pos, p0=[0.5, 0.3],
                bounds=([0, 0.01], [5.0, 3.0]), maxfev=5000
            )
            residuals_pow = acf_pos - _power_law(lags_pos, *popt_pow)
            rss_pow = float(np.sum(residuals_pow ** 2))
            n_fit = len(acf_pos)
            aic_pow = n_fit * np.log(rss_pow / n_fit + 1e-16) + 2 * 2
            beta_hat = float(popt_pow[1])
            power_law_ok = True
    except (RuntimeError, ValueError):
        pass

    exp_ok = False
    aic_exp = np.inf
    try:
        positive_mask = acf > 0
        if np.sum(positive_mask) >= 5:
            lags_pos = lags[positive_mask]
            acf_pos = acf[positive_mask]
            popt_exp, _ = curve_fit(
                _exponential, lags_pos, acf_pos, p0=[0.5, 0.1],
                bounds=([0, 0.001], [5.0, 5.0]), maxfev=5000
            )
            residuals_exp = acf_pos - _exponential(lags_pos, *popt_exp)
            rss_exp = float(np.sum(residuals_exp ** 2))
            n_fit = len(acf_pos)
            aic_exp = n_fit * np.log(rss_exp / n_fit + 1e-16) + 2 * 2
            exp_ok = True
    except (RuntimeError, ValueError):
        pass

    delta_aic = aic_exp - aic_pow

    if not has_structure_beyond_5:
        verdict = "fail"
    elif power_law_ok and delta_aic > 0 and 0.2 <= beta_hat <= 0.6:
        verdict = "pass"
    elif power_law_ok and delta_aic > 0 and not (0.2 <= beta_hat <= 0.6):
        verdict = "marginal"
    elif exp_ok and delta_aic < -2:
        verdict = "fail"
    elif exp_ok and -2 <= delta_aic < 0:
        verdict = "marginal"
    else:
        verdict = "marginal"

    return {
        "test_id": "acf_absolute_returns", "verdict": verdict,
        "polarity": "benchmark_comparison",
        "statistic": float(beta_hat) if np.isfinite(beta_hat) else None,
        "p_value": None, "p_value_bh": None,
        "benchmark": None,
        "auxiliary": {
            "aic_power_law": float(aic_pow) if np.isfinite(aic_pow) else None,
            "aic_exponential": float(aic_exp) if np.isfinite(aic_exp) else None,
            "delta_aic": float(delta_aic) if np.isfinite(delta_aic) else None,
            "power_law_fit": power_law_ok,
            "has_structure_beyond_5": bool(has_structure_beyond_5),
        },
    }

def _apply_bh_correction(
    test_results: List[Dict], alpha_fdr: float = 0.10
) -> None:
    """Apply BH FDR correction in-place to tests that contribute p-values."""
    BH_TESTS = {
        "ljung_box_raw", "ljung_box_squared", "arch_lm", "jarque_bera", "leverage"
    }
    eligible = [t for t in test_results if t["test_id"] in BH_TESTS and t["p_value"] is not None]
    if not eligible:
        return

    p_values = np.array([t["p_value"] for t in eligible])
    m = len(p_values)
    sorted_idx = np.argsort(p_values)
    sorted_p = p_values[sorted_idx]

    adjusted = np.empty(m)
    adjusted[sorted_idx[-1]] = sorted_p[-1]
    for i in range(m - 2, -1, -1):
        raw_adj = sorted_p[i] * m / (np.where(sorted_idx == sorted_idx[i])[0][0] + 1)
        adjusted[sorted_idx[i]] = min(raw_adj, adjusted[sorted_idx[i + 1]])
    adjusted = np.minimum(adjusted, 1.0)

    adjusted_simple = np.empty(m)
    ranks = np.argsort(np.argsort(p_values)) + 1
    raw_adjusted = p_values * m / ranks

    order = np.argsort(ranks)[::-1]
    running_min = 1.0
    for idx in order:
        running_min = min(running_min, raw_adjusted[idx])
        adjusted_simple[idx] = running_min

    for i, t in enumerate(eligible):
        t["p_value_bh"] = float(np.clip(adjusted_simple[i], 0, 1))

TEST_IDS = [
    "hill_tail_index", "jarque_bera", "ljung_box_raw",
    "ljung_box_squared", "arch_lm", "leverage", "acf_absolute_returns",
]


def run_test_suite(
    returns_252: np.ndarray,
    regime_labels_252: np.ndarray,
    benchmark: Dict[str, Dict],
) -> Dict:

    result: Dict = {}

    for regime_idx, regime_name in enumerate(REGIME_NAMES):
        mask = regime_labels_252 == regime_idx
        regime_ret = returns_252[mask]
        sample_count = int(np.sum(mask))

        bm = benchmark[regime_name]

        if sample_count < 20:
            # Not enough data for meaningful tests, all fail with caution
            tests_result = {}
            for tid in TEST_IDS:
                tests_result[tid] = {
                    "test_id": tid, "verdict": "fail",
                    "polarity": "N/A", "statistic": None,
                    "p_value": None, "p_value_bh": None,
                    "benchmark": None,
                    "auxiliary": {"insufficient_data": True, "sample_count": sample_count},
                }
            result[regime_name] = {"sample_count": sample_count, "tests": tests_result}
            continue

        tests: List[Dict] = [
            _test_hill(regime_ret, bm),
            _test_jarque_bera(regime_ret),
            _test_lb_raw(regime_ret),
            _test_lb_squared(regime_ret),
            _test_arch_lm(regime_ret),
            _test_leverage(regime_ret, bm),
            _test_acf_abs(regime_ret),
        ]

        _apply_bh_correction(tests)

        tests_result = {t["test_id"]: t for t in tests}
        result[regime_name] = {"sample_count": sample_count, "tests": tests_result}

    failing = []
    for regime_name in REGIME_NAMES:
        if regime_name not in result:
            continue
        for tid, t in result[regime_name]["tests"].items():
            if t["verdict"] in ("fail", "marginal"):
                failing.append({
                    "regime": regime_name, "test": tid, "verdict": t["verdict"]
                })

    return {"regime_results": result, "failing_tests_summary": failing}


def print_test_suite(suite_result: Dict) -> None:
    regime_results = suite_result["regime_results"]

    col_width = 14
    header_tests = [
        ("hill_tail_index", "Hill"),
        ("jarque_bera", "JB"),
        ("ljung_box_raw", "LB-Raw"),
        ("ljung_box_squared", "LB-Sq"),
        ("arch_lm", "ARCH-LM"),
        ("leverage", "Leverage"),
        ("acf_absolute_returns", "ACF-Abs"),
    ]

    print("\n" + "=" * 120)
    print(f"{'Regime':<14}", end="")
    for _, short_name in header_tests:
        print(f"{short_name:^{col_width}}", end="")
    print(f"  {'N':>5}")
    print("-" * 120)

    verdict_symbols = {"pass": "PASS", "marginal": "MARG", "fail": "FAIL"}

    for regime_name in REGIME_NAMES:
        rr = regime_results[regime_name]
        print(f"{regime_name:<14}", end="")
        for tid, _ in header_tests:
            t = rr["tests"][tid]
            v = verdict_symbols.get(t["verdict"], "???")
            stat = t["statistic"]
            if stat is not None:
                cell = f"{v}({stat:.2f})"
            else:
                cell = v
            print(f"{cell:^{col_width}}", end="")
        print(f"  {rr['sample_count']:>5}")

    print("=" * 120)

    failing = suite_result["failing_tests_summary"]
    n_fail = sum(1 for f in failing if f["verdict"] == "fail")
    n_marg = sum(1 for f in failing if f["verdict"] == "marginal")
    print(f"\nSummary: {n_fail} failures, {n_marg} marginal across {len(REGIME_NAMES)} regimes × {len(header_tests)} tests = {len(REGIME_NAMES) * len(header_tests)} cells")
    if n_fail > 0:
        print("\nFail details:")
        for f in failing:
            if f["verdict"] == "fail":
                print(f"  {f['regime']:>12} × {f['test']}")
    print()

def run_cvae_demo(
    train_all_dict: Dict,
    individual_returns: Dict[str, pd.Series],
) -> Dict:

    regime_labels = train_all_dict["regime_labels"]
    trans_matrix = train_all_dict["transition_matrix"]
    cvae_model = train_all_dict["cvae_model"]
    cvae_res = train_all_dict["cvae_residual_std"]

    print("=== Computing benchmark from real equity data ===")
    benchmark = compute_benchmark(individual_returns, regime_labels)
    for rn, bm in benchmark.items():
        print(f"  {rn}: Hill α={bm['hill_alpha_mean']:.2f}±{bm['hill_alpha_std']:.2f}, "
              f"L+={bm['leverage_L_plus']:.4f}±{bm['leverage_sigma']:.4f}")

    regime_to_idx = {name: i for i, name in enumerate(REGIME_NAMES)}
    regime_labels_int = regime_labels.map(regime_to_idx).dropna().astype(int)

    ticker = list(individual_returns.keys())[0]
    series = individual_returns[ticker]
    common = series.index.intersection(regime_labels_int.index)
    series_aligned = series.loc[common]
    reg_aligned = regime_labels_int.loc[common]

    seed_start = np.random.randint(0, len(series_aligned) - SEED_WINDOW)
    seed = series_aligned.values[seed_start : seed_start + SEED_WINDOW].astype(np.float32)
    initial_regime = int(reg_aligned.values[seed_start + SEED_WINDOW - 1])
    print(f"\n=== Seed: {ticker}, start idx {seed_start}, initial regime = {REGIME_NAMES[initial_regime]} ===")

    print("=== Generating 252-step CVAE sequence ===")
    regime_seq = generate_regime_sequence(trans_matrix, initial_regime, SEQUENCE_LENGTH)

    buffer = list(seed)
    for t in range(SEQUENCE_LENGTH):
        w = np.array(buffer[-WINDOW:], dtype=np.float32)
        r = cvae_generate_next(w, int(regime_seq[t]), cvae_model, cvae_res)
        if not np.isfinite(r):
            r = 0.0
        buffer.append(float(r))
    returns_252 = np.array(buffer[-SEQUENCE_LENGTH:], dtype=np.float32)

    regime_prev = {name: np.sum(regime_seq == i) / SEQUENCE_LENGTH
                   for i, name in enumerate(REGIME_NAMES)}
    print(f"  Generated sequence: std={np.std(returns_252):.6f}, "
          f"mean={np.mean(returns_252):.6f}")
    print(f"  Regime prevalence: {regime_prev}")

    std_r = np.std(returns_252)
    if std_r < 1e-8:
        print("  BLOCKED: near-zero variance")
        return {}
    if np.any(~np.isfinite(returns_252)):
        print("  BLOCKED: NaN/Inf detected")
        return {}
    if std_r > 5.0:
        print("  WARNING: extreme variance")

    print("\n=== Running Layer 2 test suite (7 tests × 3 regimes) ===")
    suite_result = run_test_suite(returns_252, regime_seq, benchmark)

    print_test_suite(suite_result)

    return suite_result

results = train_all()
individual_returns = fetch_individual_returns()
suite = run_cvae_demo(results, individual_returns)

=== Fetching VIX + TNX data ===
=== Fetching equities returns ===


/tmp/ipykernel_623/3695538259.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  vix = yf.download("^VIX", start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/3695538259.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  tnx = yf.download("^TNX", start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/622082558.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(tickers, start=start, end=end, progress=False)["Close"]


=== Fitting GMM (K=3) ===
  Regime distribution:
regime
low_vol     0.520814
mid_vol     0.416271
high_vol    0.062916
=== Estimating Markov chain ===
  Transition matrix:
[[0.98028478 0.01861993 0.00109529]
 [0.02465753 0.9716895  0.00365297]
 [0.         0.03323263 0.96676737]]
=== Building pooled training dataset ===
  Dataset: 78450 samples, window=30
=== Training GBM ===
=== Training GARCH(1,1) ===
=== Calibrating Heston ===
=== Training TFT ===
=== Training CVAE ===
=== Training Diffusion ===
=== All training complete ===


/tmp/ipykernel_623/86867600.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(t, start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/86867600.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(t, start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/86867600.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(t, start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/86867600.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(t, start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/86867600.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(t, start=start, end=end, progress=False)["Close"].squeeze()
/tmp/ipykernel_623/868676

=== Computing benchmark from real equity data ===
  low_vol: Hill α=3.38±0.48, L+=-0.0460±0.0436
  mid_vol: Hill α=3.48±0.39, L+=-0.1386±0.0454
  high_vol: Hill α=2.61±0.44, L+=-0.4993±0.0432

=== Seed: AAPL, start idx 3175, initial regime = low_vol ===
=== Generating 252-step CVAE sequence ===
  Generated sequence: std=0.014789, mean=0.001637
  Regime prevalence: {'low_vol': np.float64(0.6666666666666666), 'mid_vol': np.float64(0.3333333333333333), 'high_vol': np.float64(0.0)}

=== Running Layer 2 test suite (7 tests × 3 regimes) ===

Regime             Hill           JB          LB-Raw        LB-Sq        ARCH-LM       Leverage      ACF-Abs          N
------------------------------------------------------------------------------------------------------------------------
low_vol         FAIL(2.31)   PASS(58.98)    PASS(7.35)    FAIL(5.21)    FAIL(2.78)    FAIL(0.42)    FAIL(0.01)      168
mid_vol         FAIL(2.46)   PASS(43.62)   PASS(12.82)    FAIL(8.79)    FAIL(1.09)   PASS(-0.28) 